In [1]:
import scanpy as sc
import anndata as ad
import numpy as np
import anndata as ad

# ------------------------------------------------------------
# 1. Load the full dataset
# ------------------------------------------------------------
adata = ad.read_h5ad("multiperturb_seq.h5ad")

# ------------------------------------------------------------
# 2. Ensure ATAC_gene exists (same shortcut as before)
# ------------------------------------------------------------
if 'ATAC_gene' not in adata.obsm:
    adata.obsm['ATAC_gene'] = adata.obsm['ATAC']
    print("Copied ATAC -> ATAC_gene")

# ------------------------------------------------------------
# 3. Create 'guide_target' column
# ------------------------------------------------------------
adata.obs['guide_target'] = adata.obs['guide_gene'].apply(
    lambda x: 'NTC' if 'non-targeting' in str(x) else x
)
print("Unique guide_target:", adata.obs['guide_target'].unique())

Copied ATAC -> ATAC_gene
Unique guide_target: ['BAZ1B' 'ACTL6A' 'NTC' 'PHF10' 'GATAD2A' 'H3F3A' 'MBD3' 'SMARCA4' 'BAZ2A'
 'SMARCB1' 'MTA3' 'BPTF' 'SMARCA2' 'ARID2' 'TET2' 'DPF1' 'DNMT1' 'MLLT1'
 'CHD4' 'UCHL5' 'RBBP7' 'BRD9' 'PRMT5' 'HDAC1' 'SUZ12' 'BRD4' 'ZNHIT1'
 'BMI1' 'YBX1' 'CECR2' 'KAT2A' 'CHD1' 'CTCF' 'KMT2B' 'KDM1A' 'SIN3A'
 'DNMT3A' 'PBRM1' 'SETD2' 'SIRT1' 'MECOM' 'ATRX' 'PRDM4' 'EHMT2' 'TRRAP'
 'SUV39H2' 'SOX2' 'SETD5' 'SUV39H1' 'SMARCE1' 'SMARCC1' 'PRDM16' 'SSRP1'
 'DOT1L' 'CBX3' 'HDAC2' 'BCOR' 'DNMT3L' 'ACTR5' 'EP300' 'HAT1' 'DPF2'
 'KMT2D' 'EZH2' 'MBD1' 'GATAD2B' 'BRD7' 'SMARCA1' 'L3MBTL1' 'EZH1' 'SRCAP'
 'KDM6A' 'BRD2' 'JARID2' 'INO80' 'DRAP1' 'MTA1' 'ACTL6B' 'KAT5' 'SETDB1'
 'RBBP4' 'SIRT2' 'DNMT3B' 'MBD2' 'ZBTB18' 'BAZ1A' 'HDAC3' 'SMARCD1' 'MTA2'
 'RCOR1' 'CHD3' 'ARID1B' 'YY1' 'SS18L1' 'KMT2A' 'SMARCD2' 'SMARCC2'
 'SUPT16H' 'KMT2C' 'SETD1A']


In [2]:
# ------------------------------------------------------------
# 4. Compute pseudotime
# ------------------------------------------------------------
# Only if not already present
if 'pseudotime' not in adata.obs:
    print("Computing pseudotime...")
    if 'log1p' not in adata.uns:
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat')
    sc.tl.pca(adata, n_comps=30, use_highly_variable=True)
    pt = adata.obsm['X_pca'][:, 0]
    adata.obs['pseudotime'] = (pt - pt.min()) / (pt.max() - pt.min())
    adata.uns['pseudotime_reversed'] = False
else:
    print("Pseudotime already present.")

Computing pseudotime...


In [3]:
# ------------------------------------------------------------
# 5. Filter to NTC + SMARCE1 cells only
# ------------------------------------------------------------
subset = adata[adata.obs['guide_target'].isin(['NTC', 'SMARCE1'])].copy()
print(f"NTC cells: {(subset.obs['guide_target'] == 'NTC').sum()}")
print(f"SMARCE1 cells: {(subset.obs['guide_target'] == 'SMARCE1').sum()}")

# ------------------------------------------------------------
# 6. Sort by pseudotime (required for CCF)
# ------------------------------------------------------------
subset = subset[subset.obs['pseudotime'].argsort()].copy()

NTC cells: 25125
SMARCE1 cells: 3394


In [4]:
# ------------------------------------------------------------
# 7. Save the ready file
# ------------------------------------------------------------
subset.write("multiperturb_smarce1_ready.h5ad", compression='gzip')
print("Saved multiperturb_smarce1_ready.h5ad")

Saved multiperturb_smarce1_ready.h5ad


In [5]:
# ------------------------------------------------------------
# 8. NTC file
# ------------------------------------------------------------

# Load the file we just created
adata = ad.read_h5ad("multiperturb_smarce1_ready.h5ad")

# Keep only NTC cells
control = adata[adata.obs['guide_target'] == 'NTC'].copy()

# Save to a new file
control.write("multiperturb_ntc_only.h5ad")
print(f"NTC cells saved: {control.n_obs}")

NTC cells saved: 25125


In [6]:
import anndata as ad

# 1. Reload the original ready file (before any failed save)
adata = ad.read_h5ad("multiperturb_smarce1_ready.h5ad")

# 2. If the 'gene' column is missing, get it from the original full dataset
if 'gene' not in adata.var.columns:
    original = ad.read_h5ad("multiperturb_seq.h5ad")
    adata.var['gene'] = original.var['gene']
    
# Add ATAC_gene if missing (copy from ATAC)
if 'ATAC_gene' not in adata.obsm:
    adata.obsm['ATAC_gene'] = adata.obsm['ATAC']
    print("ATAC_gene added from ATAC")
else:
    print("ATAC_gene already present")

# 3. Set var_names from the 'gene' column, then clear the index name
adata.var_names = adata.var['gene'].astype(str)
adata.var.index.name = None
adata.var_names_make_unique()
print("First 20 var_names:", adata.var_names[:20])

# 4. Save the corrected file (overwrite)
# Quick check that everything is there
print("obsm keys:", list(adata.obsm.keys()))
print("obs columns:", list(adata.obs.columns))
print("NTC cells:", (adata.obs['guide_target'] == 'NTC').sum())
print("SMARCE1 cells:", (adata.obs['guide_target'] == 'SMARCE1').sum())

# Overwrite the file
adata.write("multiperturb_smarce1_ready.h5ad", compression='gzip')
print("Corrected file saved.")

# 5. Create target_genes_smarce1.txt
pert_genes = adata.obs['guide_gene'].unique().tolist()
with open("target_genes_smarce1.txt", "w") as f:
    for g in pert_genes:
        f.write(g + "\n")
print("Target genes written:", pert_genes[:10])

ATAC_gene already present
First 20 var_names: Index(['TSPAN6', 'TNMD', 'DPM1', 'SCYL3', 'C1orf112', 'FGR', 'CFH', 'FUCA2',
       'GCLC', 'STPG1', 'NIPAL3', 'LAS1L', 'ENPP4', 'SEMA3F', 'CFTR', 'ANKIB1',
       'CYP51A1', 'KRIT1', 'RAD52', 'BAD'],
      dtype='object')
obsm keys: ['ATAC', 'ATAC_gene', 'X_pca']
obs columns: ['guide_id', 'guide_gene', 'guide_target', 'pseudotime']
NTC cells: 25125
SMARCE1 cells: 3394
Corrected file saved.
Target genes written: ['non-targeting (human)', 'non-targeting (mouse)', 'SMARCE1']
